In [3]:
import itertools

import numpy as np
import scipy as sp
import networkx as nx
import pandas as pd

import igl
import gpytoolbox as gpy

import pyvista as pv
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import cm

from src import shapes, triangletools, vis

In [7]:
def diffeo(vertices, a=0.2, b=1.1, c=0.2, d=1.2, e=0.2, f=0.3):
    v = vertices.copy()
    x, y, z = v[:, 0], v[:, 1], v[:, 2]

    v[:, 0] = x + a*np.sin(b*y) + c*np.sin(d*z)
    v[:, 1] = y + e*np.sin(f*x)

    return v

def cylindrical_twist(vertices, k=0.2, mode="x", scale=0.5):
    """
    Nonlinear cylindrical twist diffeomorphism on R^3.

    vertices: (n,3) array
    mode:
      - "z": angle depends on z  (theta = k * tanh(z/scale))
      - "r": angle depends on radius r (theta = k * tanh(r/scale))
    k: twist strength (radians, roughly bounded by +/-k for tanh)
    scale: controls how quickly tanh saturates
    """
    if mode == 'x':
        v = vertices[:, [1, 2, 0]]
        v = cylindrical_twist(v, k=k, mode="z", scale=scale)
        v = v[:, [2, 0, 1]]
        return v
        
    v = vertices.copy()
    x, y, z = v[:, 0], v[:, 1], v[:, 2]

    r = np.sqrt(x*x + y*y)

    if mode == "z":
        theta = k * np.tanh(z / scale)
    elif mode == "r":
        theta = k * np.tanh(r / scale)
    else:
        raise ValueError('mode must be "z" or "r"')

    c, s = np.cos(theta), np.sin(theta)

    v[:, 0] = c * x - s * y
    v[:, 1] = s * x + c * y
    # z unchanged
    return v

In [8]:
n, m = 17, 16
#n, m = 57, 56

np.random.seed(42)
vertices0, faces0 = shapes.get_halftori_bouquet(leaves=3, n=n, m=m, l0=1.0, extra_points_on_edge=7, extra_points_on_disk=40, glue=True)
vertices0, faces0 = shapes.split_large_edges(vertices0, faces0, max_length=0.7)

n0, n1 = 24, 24
#n0, n1 = 48, 48
r0, r1 = 0.8, 1.1
vertices1, faces1 = shapes.get_couple_linked_tori(n0=n0, n1=n1, r0=r0, r1=r1)

vertices0 += np.array([vertices0[:, 0].min(), 0, 0])
vertices1 += np.array([vertices1[:, 0].max(), 0, 0])
vertices1, faces1 = shapes.split_large_edges(vertices1, faces1, max_length=0.8)

vertices, faces, _ = shapes.merge_meshes_with_weld(vertices0, faces0, vertices1, faces1)
#vertices, faces = shapes.split_large_edges(vertices, faces, max_length=0.8)

vertices = cylindrical_twist(vertices, k=-0.6, scale=3.2, mode='x')

In [9]:
mesh = vis.get_pv_mesh(vertices, faces)

pl = pv.Plotter(window_size=(600, 600))
pl.add_mesh(mesh, color='white', smooth_shading=False, show_edges=True, opacity=1.0)
pl.show()

Widget(value='<iframe src="http://localhost:39879/index.html?ui=P_0x795ec206c560_0&reconnect=auto" class="pyvi…

In [10]:
import numpy as np

def save_ply(filename, vertices, faces):
    with open(filename, "w") as f:
        # header
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(vertices)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write(f"element face {len(faces)}\n")
        f.write("property list uchar int vertex_indices\n")
        f.write("end_header\n")

        # vertices
        for v in vertices:
            f.write(f"{v[0]} {v[1]} {v[2]}\n")

        # faces
        for face in faces:
            f.write(f"3 {face[0]} {face[1]} {face[2]}\n")

In [11]:
save_ply('data/nm-mesh.ply', vertices, faces)

In [14]:
import trimesh

mesh = trimesh.load("data/nm-mesh-iso.ply")

vertices = mesh.vertices      # shape (n,3)
faces = mesh.faces            # shape (k,3)

print(vertices.shape, faces.shape)

(11364, 3) (22984, 3)


In [15]:
mesh = vis.get_pv_mesh(vertices, faces)

pl = pv.Plotter(window_size=(600, 600))
pl.add_mesh(mesh, color='white', smooth_shading=False, show_edges=True, opacity=1.0)
pl.show()

Widget(value='<iframe src="http://localhost:39879/index.html?ui=P_0x795f06f1b170_1&reconnect=auto" class="pyvi…